# CKD EDA — read-only profiling notebook

**Read-only contract (Pattern N1):** this notebook observes the raw CKD data through the existing pipeline functions only.
It imports `load_raw_data`, `summarize`, and `clean_raw` from `src.*` and never re-implements quirk handling
(`?` to NaN, whitespace stripping, dtype coercion). It never fills, imputes, encodes, or fits anything —
working copies are named `eda_df` / `plot_df` and are only filtered or coerced for plotting.
Every missingness number shown here comes from `summarize()`, the single source of truth.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # kernel cwd is notebooks/ -> repo root (Pattern N1 shim)

ROOT = Path("..").resolve() if Path("..", "src").exists() else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ingestion.load_data import load_raw_data, summarize
from src.preprocessing.preprocess import NUMERIC_COLS, CATEGORICAL_COLS, clean_raw

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
eda_df = load_raw_data()  # configured raw path; quirks already normalized upstream

summary = summarize(eda_df)  # single source of truth for missingness numbers
summary

In [ ]:
missing_pct = eda_df.isna().mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
missing_pct.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("missing %")
ax.set_title("Per-column missingness (raw CKD data)")
fig.savefig(FIG_DIR / "missingness_bar.png", bbox_inches="tight")
plt.close(fig)
missing_pct

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(eda_df.isna(), cbar_kws={"label": "missing"}, yticklabels=False, ax=ax)
ax.set_title("Missing-value map (yellow = missing)")
fig.savefig(FIG_DIR / "missingness_heatmap.png", bbox_inches="tight")
plt.close(fig)

## Target-aware distributions

`plot_df` reuses the pipeline's own `clean_raw` cleaning (target labels mapped to 0/1 by the pipeline,
no manual recoding here). Hemoglobin shows class separation; red-blood-cell counts are compared per class.

In [ ]:
plot_df = clean_raw(load_raw_data())  # reuse pipeline cleaning — never recode labels by hand

grid = sns.displot(plot_df, x="hemo", hue="classification", kind="kde", fill=True)
grid.fig.suptitle("Hemoglobin distribution by CKD class", y=1.02)
grid.fig.savefig(FIG_DIR / "hemo_kde_by_class.png", bbox_inches="tight")
plt.close("all")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=plot_df, x="rbc", hue="classification", ax=ax)
ax.set_title("RBC counts by CKD class")
fig.savefig(FIG_DIR / "rbc_count_by_class.png", bbox_inches="tight")
plt.close(fig)

## Numeric correlations

Numeric columns coerced with `pd.to_numeric(errors="coerce")` (mirroring `clean_raw`), Pearson matrix
on a fixed [-1, 1] diverging scale with the upper triangle masked.

In [ ]:
num = eda_df[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce")
corr = num.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Numeric feature correlations (Pearson)")
fig.savefig(FIG_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.close(fig)
corr

## Takeaways

1. Highest missingness is in `rbc` (38.00%), followed by `rc` (32.75%) and `wc` (26.50%) —
   all from the red/white blood-cell block, which motivates the pipeline's median/most-frequent imputation.
2. Strongest numeric correlation is hemoglobin–packed-cell-volume (Pearson r ~ 0.90); the anemia block
   (hemo, pcv, rc) is highly intercorrelated, so the model will see redundant red-cell signals.